In [1]:
import os

os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")
from langchain.chat_models import init_chat_model
# Grog model

model = init_chat_model("groq:qwen/qwen3-32b")

/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [16]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, SystemMessage

In [11]:
agent=create_agent(
    model ="groq:qwen/qwen3-32b",
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="groq:qwen/qwen3-32b",
            trigger=("messages", 10),
            keep =("messages", 4)
        )
    ]
)

In [12]:
#run with threadid
config ={"configurable":{"thread_id":"test-1"}}

In [14]:
#test data
questions=[
    "whats is 2+2?",
    "whats is 3*2?",
    "whats is 100/5?",
    "whats is 15-9?",
    "whats is 4*8?",

]

for q in questions:
    response =agent.invoke({"messages":[HumanMessage(content=q)]},config)
    print(f"messages: {response}")
    print(f"messages: {len(response['messages'])}")



messages: {'messages': [HumanMessage(content='Here is a summary of the conversation to date:\n\n<think>\nOkay, let me tackle this. The user asked, "whats is 15-9?" So, subtraction of 9 from 15.\n\nFirst, basic arithmetic. 15 minus 9. Let me think: 9 plus 6 is 15, so 15 minus 9 should be 6. Alternatively, counting down from 15 by 9 steps. 15-9=6. \n\nWait, maybe another way to check. If I take 9 away from 15, how much is left? 15 - 9 = 6. That\'s correct. \n\nIs there any alternative method? Maybe using number lines: start at 15, move back 9 units. 15 to 6 is 9 steps. So yes, 6. \n\nNo tricks here. The question is straightforward. The answer should be 6. No need for complicated explanations. Just confirm the subtraction fact. The user likely wants a quick answer. \n\nNo context given, so standard arithmetic applies. The answer is 6.\n</think>\n\nThe result of subtracting 9 from 15 is **6**. This can be confirmed by recognizing that 9 + 6 = 15, so 15 - 9 = 6. \n\n**Answer:** 6.', additio

In [18]:
#token size
from langchain_core.tools import tool
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, SystemMessage

@tool
def search_hotels(city:str)->str:
      """Search hotels - returns long response to use more tokens."""
      return f"""Hotels in {city}:
      1. Grand Hotel - 5 star, $350/night, spa, pool, gym
      2. City Inn - 4 star, $180/night, business center
      x 3. Budget Stay - 3 star, $75/night, free wifi"""

agent= create_agent(
    model ="groq:qwen/qwen3-32b",
    tools=[search_hotels],
    checkpointer =InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="groq:qwen/qwen3-32b",
            trigger=("tokens", 550),
            keep=("tokens", 200)
        )
    ]

)

config ={"configurable":{"thread_id":"test-1"}}

In [2]:
#token counter

def count_tokens(messages):
    total_chars= sum(len(str(m.content)) for m in messages)
    return total_chars 

In [23]:
#run test

cities=["paris", "London", "Tokyo", "Dubai", "new York", "singapore"]

for city in cities:
    response = agent.invoke(
        {"messages":[HumanMessage(content=f"find hotels in {city}")]},config =config
    )

tokens=count_tokens(response['messages'])
print(f"{city}: ~{tokens} tokens, {len(response['messages'])} messages")
print(f"{response['messages']}")

singapore: ~3195 tokens, 5 messages
[HumanMessage(content='Here is a summary of the conversation to date:\n\n<think>\nOkay, let\'s see. The user provided a conversation history where the AI needs to extract key context. The main user intent is to find hotels in different cities, but Budget Stay is repeatedly marked with an "x" in multiple cities. The assistant has to structure the summary into four sections: Session Intent, Summary, Artifacts, and Next Steps.\n\nFirst, the Session Intent is straightforward: the user is looking for hotel options in various cities, and there\'s a recurring issue with Budget Stay being flagged. The Summary needs to list the hotels provided for New York, mention the x-flag, and note the pattern from previous cities. Artifacts would include the hotel search results and the historical x-flags. Next Steps involve the user deciding on a hotel or investigating the x-flag.\n\nI need to make sure each section is concise but captures all critical points. The x-fla

singapore: ~3873 tokens, 5 messages
[HumanMessage(content='Here is a summary of the conversation to date:\n\n<think>\nOkay, let me try to figure out what the user wants here. They provided a conversation history where the assistant needs to extract the most relevant context. The user\'s latest query is about finding hotels in New York, following previous searches for Tokyo and Dubai. \n\nFirst, I need to identify the session intent. The user\'s primary goal now is to find hotels in New York. The previous searches for Tokyo and Dubai are part of the history but the current focus is New York.\n\nNext, the summary should capture the key points from the conversation. The assistant previously listed Dubai hotels and now needs to present New York options. The user hasn\'t specified any criteria yet, so the summary should mention that the search is ongoing and that the assistant is ready to refine based on user input.\n\nFor artifacts, the Dubai hotel list is a historical artifact, and the Ne

In [ ]:
#fraction

#token size
from langchain_core.tools import tool
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, SystemMessage

@tool
def search_hotels(city:str)->str:
      """Search hotels - returns long response to use more tokens."""
      return f"""Hotels in {city}:
      1. Grand Hotel - 5 star, $350/night, spa, pool, gym
      2. City Inn - 4 star, $180/night, business center
      x 3. Budget Stay - 3 star, $75/night, free wifi"""

agent= create_agent(
    model ="groq:qwen/qwen3-32b",
    tools=[search_hotels],
    checkpointer =InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="groq:qwen/qwen3-32b",
            trigger=("fraction", 0.005),
            keep=("fraction", 0.002)
        )
    ]

)

config ={"configurable":{"thread_id":"test-1"}}

def count_tokens(messages):
    total_chars= sum(len(str(m.content)) for m in messages)
    return total_chars 

cities=["paris", "London", "Tokyo", "Dubai", "new York", "singapore"]

for city in cities:
    response = agent.invoke(
        {"messages":[HumanMessage(content=f"find hotels in {city}")]},config =config
    )

tokens=count_tokens(response['messages'])
print(f"{city}: ~{tokens} tokens ({fraction:.4%}), {len(response['messages'])} messages")
print(f"{response['messages']}")